In [1]:
from pyspark.sql.types import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.sql import SparkSession

# Stop any existing session first
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("TaxiPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

In [2]:

schema = StructType([

    StructField("VendorID", IntegerType(),True),
    StructField("tpep_pickup_datetime", TimestampType(),True),
    StructField("tpep_dropoff_datetime", TimestampType(),True),
    StructField("passenger_count", IntegerType(),False),
    StructField("trip_distance", FloatType(),False),
    StructField("pickup_longitude", FloatType(),False),
    StructField("pickup_latitude", FloatType(),False),
    StructField("RateCodeID", IntegerType(),True),
    StructField("store_and_fwd_flag", StringType(),True),
    StructField("dropoff_longitude", FloatType(),True),
    StructField("dropoff_latitude", FloatType(),True),
    StructField("payment_type", IntegerType(),False),
    StructField("fare_amount", FloatType(),True),
    StructField("extra", FloatType(),True),
    StructField("mta_tax", FloatType(),True),
    StructField("tip_amount", FloatType(),True),
    StructField("tolls_amount", FloatType(),True),
    StructField("improvement_surcharge", FloatType(),True),
    StructField("total_amount", FloatType(),False),
])

In [3]:
df = spark.read.csv("../data/raw/", header=True, schema=schema)

In [4]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|RateCodeID|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|        -73.9939|       40.75011|         1|                 N|       -73.974

In [5]:

df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: float (nullable = true)
 |-- pickup_longitude: float (nullable = true)
 |-- pickup_latitude: float (nullable = true)
 |-- RateCodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: float (nullable = true)
 |-- dropoff_latitude: float (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- extra: float (nullable = true)
 |-- mta_tax: float (nullable = true)
 |-- tip_amount: float (nullable = true)
 |-- tolls_amount: float (nullable = true)
 |-- improvement_surcharge: float (nullable = true)
 |-- total_amount: float (nullable = true)



In [6]:
from pyspark.sql.functions import col


def filter_passengers(df):
    return df.filter(
        col("passenger_count").between(1, 6)
    )


def filter_fares(df):
    return df.filter(
        col("fare_amount").between(3, 500)
    )


def filter_distance(df):
    return df.filter(
        col("trip_distance").between(0.1, 50)
    )


def filter_geography(df):
    return df.filter(
        col("pickup_latitude").between(40.4, 41.2) &
        col("dropoff_latitude").between(40.4, 41.2) &
        col("pickup_longitude").between(-74.3, -73.7) &
        col("dropoff_longitude").between(-74.3, -73.7)
    )


def filter_tips(df):
    return df.filter(
        col("tip_amount").between(0, 100)
    )


def filter_surcharges(df):
    return df.filter(
        col("extra").between(0, 1) &
        col("tolls_amount").between(0, 40)
    )

In [7]:
df = (
    df.transform(filter_passengers)
      .transform(filter_fares)
      .transform(filter_distance)
      .transform(filter_geography)
      .transform(filter_tips)
      .transform(filter_surcharges)
)

In [8]:
df.describe().show()

+-------+------------------+------------------+------------------+-------------------+--------------------+------------------+------------------+--------------------+-------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+---------------------+------------------+
|summary|          VendorID|   passenger_count|     trip_distance|   pickup_longitude|     pickup_latitude|        RateCodeID|store_and_fwd_flag|   dropoff_longitude|   dropoff_latitude|       payment_type|       fare_amount|              extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|
+-------+------------------+------------------+------------------+-------------------+--------------------+------------------+------------------+--------------------+-------------------+-------------------+------------------+-------------------+-------------------+------------------+------------------+-------

In [10]:
from pyspark.sql.functions import col

null_counts = {c: df.filter(col(c).isNull()).count() for c in df.columns}
print(null_counts)

{'VendorID': 0, 'tpep_pickup_datetime': 0, 'tpep_dropoff_datetime': 0, 'passenger_count': 0, 'trip_distance': 0, 'pickup_longitude': 0, 'pickup_latitude': 0, 'RateCodeID': 0, 'store_and_fwd_flag': 0, 'dropoff_longitude': 0, 'dropoff_latitude': 0, 'payment_type': 0, 'fare_amount': 0, 'extra': 0, 'mta_tax': 0, 'tip_amount': 0, 'tolls_amount': 0, 'improvement_surcharge': 0, 'total_amount': 0}


In [11]:
from pyspark.sql.functions import month, year

df = df.withColumn("pickup_month", month(col("tpep_pickup_datetime"))) \
       .withColumn("pickup_year", year(col("tpep_pickup_datetime")))

In [ ]:
df = df.repartition(8)

df.write \
  .partitionBy("pickup_year", "pickup_month") \
  .mode("overwrite") \
  .parquet("../output/cleaned/")

In [14]:
df_verify = spark.read.parquet("../output/cleaned/")

df_verify.groupBy("pickup_year", "pickup_month") \
         .count() \
         .orderBy("pickup_year", "pickup_month") \
         .show()

+-----------+------------+--------+
|pickup_year|pickup_month|   count|
+-----------+------------+--------+
|       2015|           1|12395979|
|       2016|           1|10648458|
|       2016|           2|11077748|
|       2016|           3|11885132|
+-----------+------------+--------+

